# 02 — WDC Schema.org Download

Web Data Commons has already extracted all schema.org markup from Common Crawl.
We download their class-specific N-Quads files to get (URL, JSON-LD) training pairs.

**Key insight**: WDC gives us millions of real-world examples with zero scraping.

**Outputs**:
- `data/raw/wdc/` — downloaded .nq.gz files per schema type
- `data/raw/wdc_ie_records.jsonl` — .ie-domain records only
- `data/raw/wdc_global_records.jsonl` — global high-quality records for training

In [1]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

WDC_DIR = Path('../data/raw/wdc')
WDC_DIR.mkdir(parents=True, exist_ok=True)

from src.wdc import WDC_SUBSETS, download_wdc_subset, load_and_filter_wdc
print('WDC subsets available:', list(WDC_SUBSETS.keys()))

WDC subsets available: ['LocalBusiness', 'Product', 'Organization', 'Article', 'NewsArticle', 'BlogPosting', 'Event', 'FAQPage', 'Recipe', 'Person', 'WebSite', 'BreadcrumbList']


## Download WDC Subsets

Download the class-specific N-Quads files. These are large (~1-5GB each compressed).
Start with `LocalBusiness` and `Product` as they cover most Irish SME sites.

In [2]:
# Priority order for Irish web use case
PRIORITY_TYPES = [
    'LocalBusiness',  # Restaurants, shops, hotels — core Irish SME market
    'Product',        # E-commerce
    'Organization',   # Businesses, NGOs, gov
    'Article',        # News, blogs
    'Event',          # Events, festivals
    'FAQPage',        # FAQ sections
    'Recipe',         # Food sites
]

# Download LocalBusiness first (highest priority, largest dataset)
downloaded = {}
for schema_type in PRIORITY_TYPES[:2]:  # Start with first 2; add more as needed
    try:
        path = download_wdc_subset(schema_type, str(WDC_DIR))
        downloaded[schema_type] = path
        print(f'Downloaded {schema_type}: {path}')
    except Exception as e:
        print(f'Failed to download {schema_type}: {e}')


KeyboardInterrupt



## Parse and Filter .ie Domain Records

In [ ]:
from src.wdc import load_and_filter_wdc

ie_records = []

for schema_type, filepath in downloaded.items():
    if filepath is None:
        continue
    
    records = load_and_filter_wdc(
        filepath=str(filepath),
        schema_type=schema_type,
        tld_filter='ie',       # Only .ie domains
        min_properties=5,       # Quality filter
        max_records=10_000,     # Cap per type for exploration
    )
    ie_records.extend(records)
    print(f'{schema_type}: {len(records)} .ie records')

print(f'\nTotal .ie records: {len(ie_records)}')

In [ ]:
# Save .ie records
out_path = Path('../data/raw/wdc_ie_records.jsonl')
with open(out_path, 'w') as f:
    for rec in ie_records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'Saved {len(ie_records)} .ie records to {out_path}')

## Parse Global High-Quality Records (Training Data)

For training we want diverse, high-quality examples — not just Irish sites.
Filter globally for richness (10+ properties) to get the best training pairs.

In [ ]:
global_records = []

for schema_type, filepath in downloaded.items():
    if filepath is None:
        continue
    
    records = load_and_filter_wdc(
        filepath=str(filepath),
        schema_type=schema_type,
        tld_filter=None,        # Global
        min_properties=10,      # Higher quality threshold for training
        max_records=50_000,     # More for training
    )
    global_records.extend(records)
    print(f'{schema_type}: {len(records)} global high-quality records')

print(f'\nTotal global training records: {len(global_records)}')

In [ ]:
# Save global records
out_path = Path('../data/raw/wdc_global_records.jsonl')
with open(out_path, 'w') as f:
    for rec in global_records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'Saved {len(global_records)} global records to {out_path}')

## Analysis: Schema Quality Distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(ie_records)
print('IE records schema types:')
print(df['schema_type'].value_counts())
print(f'\nAverage property count: {df["property_count"].mean():.1f}')

df['property_count'].hist(bins=30, figsize=(10, 4))
plt.xlabel('Property Count')
plt.ylabel('Count')
plt.title('Property Count Distribution (.ie records)')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-reference: .ie domains with schema vs without
# Load CC domain list from notebook 01
cc_domain_path = Path('../data/raw/ie_domains_cc_api.jsonl')
if cc_domain_path.exists():
    cc_domains = set()
    with open(cc_domain_path) as f:
        for line in f:
            rec = json.loads(line)
            cc_domains.add(rec.get('_domain', ''))

    wdc_domains = set()
    for rec in ie_records:
        from urllib.parse import urlparse
        domain = urlparse(rec.get('source_url', '')).netloc.lstrip('www.')
        wdc_domains.add(domain)

    domains_with_schema = cc_domains & wdc_domains
    domains_without_schema = cc_domains - wdc_domains

    print(f'Total .ie domains in CC: {len(cc_domains)}')
    print(f'Domains WITH schema: {len(domains_with_schema)} ({100*len(domains_with_schema)/len(cc_domains):.1f}%)')
    print(f'Domains WITHOUT schema: {len(domains_without_schema)} ({100*len(domains_without_schema)/len(cc_domains):.1f}%)')
    print('\nDomains without schema are our primary generation targets!')
else:
    print('Run notebook 01 first to generate CC domain list')

print('\nNext step: 03_warc_extraction.ipynb')